# Boxing punch detection: full pipeline

Runs the whole process described in the project README by calling the `boxing.*` modules with `!` (shell) and `%` (magic) commands. Each stage writes its output to disk, so any cell can be rerun on its own.

| stage | module | output |
|---|---|---|
| 1. Clips | `boxing.preprocess` | `punch-seq/` |
| 2. Keypoints | `boxing.keypoints` | `punch-keypoints-boxers/` |
| 3. Pairing | `boxing.pairing` | `punch-pairs/` |
| 4-5. Train EPAM-Net | `boxing.train_punch_classifier` | `runs/epam-net/` |

Set `TEST = True` for a quick end-to-end check (a few tasks, tiny training run) before a full run.

## 0. Setup

In [ ]:
import os, json
from pathlib import Path

# Work from the repository root (this notebook lives in notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
%cd {ROOT}

TEST = False   # True: quick smoke run. False: full dataset and full training.

RAW       = "data/raw/punch"
SEQ       = "data/processed/punch-seq-test" if TEST else "data/processed/punch-seq"
KEYPOINTS = "data/processed/punch-keypoints-boxers-test" if TEST else "data/processed/punch-keypoints-boxers"
PAIRS     = "data/processed/punch-pairs-test" if TEST else "data/processed/punch-pairs"
RUN_DIR   = "runs/epam-test" if TEST else "runs/epam-net"

PREPROCESS_FLAGS = "--test" if TEST else ""
# Official EPAM-Net backbones (NTU RGB+D 120, X-Sub), see boxing/pretrained.py. False: train both streams from scratch.
USE_PRETRAINED   = True
PRETRAINED       = "data/processed/pretrained/NTU-RGB+D-120"
INIT_FLAGS = (
    f"--init-rgb {PRETRAINED}/ntu120_xsub_x3dTShiftD_I3Dhead_RGB_epoch_205.pth "
    f"--init-pose {PRETRAINED}/ntu120_xsub_x3dTShiftPose_double_shifted_chs_best_top1_acc_epoch_235.pth"
    if USE_PRETRAINED else ""
)
TRAIN_FLAGS      = ("--test" if TEST else "--epochs 20 --batch-size 16 --accum-steps 2") + INIT_FLAGS

WORKERS = os.cpu_count() or 1   # processes for the clips and pairing stages (--workers); 1 = single process, 0 = all cores
KP_WORKERS = 3                  # keypoints: each worker loads its own model; keep at 1 on a single GPU, raise it on CPU
KP_BATCH = 32                   # keypoints: images per forward pass (--batch-size); 1 = one by one

In [ ]:
from dotenv import load_dotenv

# Read the project-level .env (Kaggle and W&B credentials). Existing shell variables take precedence.
ENV_FILE = ROOT / ".env"
if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=False)
    print("Loaded", ENV_FILE)
else:
    print("No .env at", ENV_FILE, "- copy .env.example and fill it in.")

# Report which credentials are set, without printing their values
for name in ("KAGGLE_API_TOKEN", "HF_TOKEN", "WANDB_API_KEY"):
    print(f"{name:<18}", "set" if os.environ.get(name) else "missing")

Downloads the Olympic Boxing Punch Classification Video Dataset from Kaggle and unpacks it into `data/raw/punch/`. Credentials are loaded from the project `.env` (see the previous cell): `KAGGLE_API_TOKEN`, or `HF_TOKEN`. They are sent as an HTTP header and never printed. If `data/raw/punch/` already holds the task folders, the download is skipped.

In [ ]:
import os, shutil, zipfile, base64, urllib.request
from pathlib import Path

DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/piotrstefaskiue/olympic-boxing-punch-classification-video-dataset"
DOWNLOAD_DIR = ROOT / "data" / "downloads"
ZIP_PATH = DOWNLOAD_DIR / "olympic-boxing-punch.zip"
EXTRACT_DIR = DOWNLOAD_DIR / "olympic-boxing-punch"

def task_dirs(base):
    return sorted({p.parent for p in Path(base).rglob("annotations.json")})

if task_dirs(RAW):
    print("Dataset already present:", len(task_dirs(RAW)), "tasks in", RAW)
else:
    token = os.environ.get("KAGGLE_API_TOKEN")
    user, key = os.environ.get("KAGGLE_USERNAME"), os.environ.get("KAGGLE_KEY")
    if token:
        auth = f"Bearer {token}"
    elif user and key:
        auth = "Basic " + base64.b64encode(f"{user}:{key}".encode()).decode()
    else:
        raise RuntimeError("Set KAGGLE_API_TOKEN (or KAGGLE_USERNAME and KAGGLE_KEY) in .env, then rerun this cell.")

    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(DATASET_URL, headers={"Authorization": auth})
    print("Downloading (this is a large file)...")
    with urllib.request.urlopen(req) as resp, open(ZIP_PATH, "wb") as out:
        shutil.copyfileobj(resp, out, length=1024 * 1024)
    print("Saved", ZIP_PATH, f"({ZIP_PATH.stat().st_size / 1e6:.0f} MB)")

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

    # The archive layout may wrap the task folders in extra directories, so move each task folder by its own name
    Path(RAW).mkdir(parents=True, exist_ok=True)
    moved = 0
    for d in task_dirs(EXTRACT_DIR):
        dst = Path(RAW) / d.name
        if not dst.exists():
            shutil.move(str(d), str(dst))
            moved += 1
    print("Moved", moved, "task folders into", RAW, "| total:", len(task_dirs(RAW)))
    assert task_dirs(RAW), "No annotations.json found in the archive; check the archive layout in " + str(EXTRACT_DIR)

In [ ]:
# Environment check
!uv --version
!uv run python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"
!ls {RAW} | head -5

## Install the boxing package

Installs the `boxing` package from GitHub into this kernel so the `boxing.*` modules can be imported. The torch and torchvision versions already in the kernel are pinned with a constraint file, so pip does not download or replace them. This requires torch to be installed already. Rerun after upstream changes.

In [ ]:
import sys, subprocess
import os
from importlib.metadata import distribution, version, PackageNotFoundError
from pathlib import Path

# Pin the installed torch and torchvision so pip keeps them and skips the large download
pins = []
for pkg in ("torch", "torchvision"):
    try:
        pins.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        raise RuntimeError(f"{pkg} is not installed in this kernel. Install it first, then rerun this cell.")

constraints = Path("torch-constraints.txt")
constraints.write_text("\n".join(pins) + "\n")
print("Pinned:", pins)

%pip install --constraint {constraints} git+https://github.com/jmanuelc87/boxing.git@experiments/boxer-swap-and-params-configs
%pip install onnx onnxruntime-gpu

# Record the installed commit so the training run can log it to W&B (read by boxing.train_punch_classifier)
import json
direct = distribution("boxing").read_text("direct_url.json")
commit = json.loads(direct).get("vcs_info", {}).get("commit_id") if direct else None
if commit:
    os.environ["BOXING_COMMIT"] = commit
print("boxing commit:", commit or "unknown")

## 1. Clips

Interpolates the CVAT boxes between keyframes, cuts one 16-frame clip per punch track, extracts the frames as 640x360 JPEGs, and splits by recording session into `train.jsonl` / `val.jsonl`.

In [ ]:
!uv run python -m boxing.preprocess {RAW} {SEQ} {PREPROCESS_FLAGS} --workers {WORKERS}

In [ ]:
labels = json.load(open(f"{SEQ}/labels.json"))
print("seq_len:", labels["seq_len"])
for i, n in enumerate(labels["names"]):
    print(i, n)
!wc -l {SEQ}/train.jsonl {SEQ}/val.jsonl

In [ ]:
# Look at one clip: frames with the punch box drawn on them
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

rec = json.loads(open(f"{SEQ}/train.jsonl").readline())
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
for ax, fr in zip(axes.flat, rec["frames"][::2]):
    im = Image.open(f"{SEQ}/{fr['image']}").convert("RGB")
    d = ImageDraw.Draw(im)
    for (cx, cy, w, h) in fr["boxes"]:
        W, H = im.size
        d.rectangle([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H], outline="yellow", width=3)
    ax.imshow(im); ax.axis("off")
fig.suptitle(f"{rec['task']} | class: {labels['names'][rec['label']]}")
plt.tight_layout(); plt.show()

## 2. Pose keypoints

Runs the COCO pose model on every frame and keeps only the red and blue boxers (using `data/processed/role-classifier/role_model.pt`). Weights are downloaded from the Hugging Face Hub on first use (non-commercial licence).

Frames are sent through the model in batches of `KP_BATCH` (default 8), which gave about 2x the throughput of one image at a time on a 48-frame sample (`python -m boxing.bench_keypoints` reproduces this). Each of the `KP_WORKERS` processes loads its own model and runs its own batches.

Options:
- `--batch-size B`: images per forward pass; `1` processes one image at a time.
- `--workers N`: processes (`0` = all cores). Keep it at 1 on a single GPU.
- `--all-people`: keep everyone, not just the red and blue boxers.
- `--color-rules`: use the torso-colour heuristic instead of the role model.
- `--backend rtdetr`: use the RF-DETR ONNX model instead of the default YOLO pose model (`--weights`, default `yolo11m-pose.pt`).


In [ ]:
!uv run python -m boxing.keypoints {SEQ} {KEYPOINTS} --device cuda --workers {KP_WORKERS} --batch-size {KP_BATCH}

In [ ]:
kp_files = sorted(Path(KEYPOINTS, "images").rglob("*.json"))
print(len(kp_files), "keypoint files")
print(json.dumps(json.load(open(kp_files[0])), indent=1)[:800])

## 3. Paired windows

Picks the attacker (the boxer whose glove box contains the punch) and the opponent (nearest other person), and follows both through the 16 frames. Output `X` has shape `(N, 2, 16, 17, 3)`.

In [ ]:
!uv run python -m boxing.pairing --src {SEQ} \
    --keypoints {KEYPOINTS} --dst {PAIRS} --workers 6

In [ ]:
import numpy as np
from collections import Counter

for split in ("train", "val"):
    d = np.load(f"{PAIRS}/{split}.npz")
    meta = [json.loads(l) for l in open(f"{PAIRS}/{split}.jsonl")]
    print(split, "X", d["X"].shape, "y", d["y"].shape, dict(Counter(m["quality"] for m in meta)))

## 4-5. Crop, augment and train EPAM-Net

Cropping to both boxers (224x224), augmentation, normalised pose heatmaps and the two-stream model all run inside the training command. With `USE_PRETRAINED = True` (setup cell) both backbones start from the official EPAM-Net NTU RGB+D 120 X-Sub weights; the pose stem is copied for both boxers. The heads and the attention block are trained from scratch. Check the pipeline with `--test` first (set `TEST = True` above); it also reloads the checkpoint and fails if the scores differ.

### Training flags

`TRAIN_FLAGS` (setup cell) is passed to `boxing.train_punch_classifier`. Every flag, its script default and what it does. The **Notebook** column shows this notebook's value; "default" means the script default is used. Full reference: `docs/training-flags.md`.

| Flag | Default | Notebook | Description |
|---|---|---|---|
| `--pairs PATH` | `data/processed/punch-pairs` | `PAIRS` | Output of `boxing.pairing`: windows, keypoints and labels. |
| `--src PATH` | `data/processed/punch-seq` | `SEQ` | Output of `boxing.preprocess`: frames and clips for the RGB stream. |
| `--out PATH` | `runs/epam-net` | `RUN_DIR` | Where the best model, checkpoint, model card and `metrics.json` go. |
| `--epochs N` | `20` | `20` | Maximum epochs. Training can stop earlier via `--patience`. |
| `--batch-size N` | `16` | `16` | Clips per forward pass. BatchNorm statistics are computed over them. |
| `--accum-steps N` | `2` | `2` | Batches per optimiser step; effective batch = batch size × this (32). |
| `--freeze-bn` | off | off | Keep BatchNorm running statistics fixed; affine parameters still train. |
| `--lr X` | `1e-3` | `1e-3` | Peak AdamW learning rate for heads and attention. |
| `--backbone-lr X` | `1e-4` | `1e-4` | Peak AdamW learning rate for the RGB and pose backbones. |
| `--weight-decay X` | `0.01` | `0.1` | AdamW decoupled weight decay; not applied to BatchNorm weights and biases. |
| `--warmup-epochs X` | `2.0` | default | Linear warm-up length in epochs, then cosine decay to 1% of peak. |
| `--grad-clip X` | `5.0` | default | Clip gradient norm before each step; `0` disables. |
| `--loss {ce,focal}` | `focal` | default | Focal loss, or class-weighted cross entropy with `ce`. |
| `--focal-gamma X` | `1.0` | default | Focusing parameter, used with `--loss focal`. |
| `--label-smoothing X` | `0.05` | default | Label smoothing in the loss. |
| `--rgb-weight X` | `0.5` | default | RGB stream's weight when fusing the two streams' scores; pose gets `1 - X`. |
| `--dropout X` | `0.5` | default | Dropout before each classifier head. |
| `--min-score X` | `0.1` | default | Joints at or below this keypoint confidence are left out of the pose heatmaps. |
| `--init-rgb PATH` | none | `INIT_FLAGS` | Official EPAM-Net RGB checkpoint for the RGB backbone. |
| `--init-pose PATH` | none | `INIT_FLAGS` | Official EPAM-Net pose checkpoint for the pose backbone. |
| `--sigma VALUE` | `0.6` | default | Gaussian radius of pose heatmaps; one value, or 17 comma-separated per joint. |
| `--no-swap` | off | off | Turn off the random swap of the two boxers' keypoint rows (training augmentation). |
| `--patience N` | `10` | default | Stop after N epochs without a validation macro-F1 gain. |
| `--quality {all,good}` | `all` | default | Which paired windows to use; `good` drops `partial` windows. |
| `--workers N` | `4` | default | Data-loading processes. |
| `--device NAME` | cuda, else cpu | default | Torch device. |
| `--seed N` | `0` | default | Random seed. |
| `--test` | off | `TEST` | Smoke test on a few windows, 2 epochs, no W&B or Hub. |
| `--wandb-project NAME` | `boxing` | default | Weights & Biases project (needs `WANDB_API_KEY`). |
| `--wandb-name NAME` | wandb picks | default | W&B run name. |
| `--log-every N` | `100` | default | Push step loss and LR to W&B every N optimiser steps. |
| `--no-wandb` | off | default | Disable W&B logging. |
| `--hub-repo REPO` | `jmanuelc87/boxing-punch-epam-net` | default | Hub repo for the best model and checkpoints. |
| `--hub-public` | off | default | Create the Hub repo public if missing (default private). |
| `--no-push` | off | default | Do not upload to the Hub. |

In [ ]:
!uv run python -m boxing.train_punch_classifier --pairs {PAIRS} --src {SEQ} --out {RUN_DIR} {TRAIN_FLAGS} --quality all --sigma 0.6 --focal-gamma 2 --weight-decay 0.1 --grad-clip 23.0 --backbone-lr 1e-4

## Results

In [ ]:
m = json.load(open(f"{RUN_DIR}/metrics.json"))
print(list(m.keys()))
!ls -la {RUN_DIR}

In [ ]:
hist = m.get("history", [])
if hist:
    import pandas as pd
    df = pd.DataFrame(hist)
    display(df.tail(10))
    num = df.select_dtypes("number")
    num.plot(subplots=True, figsize=(8, 2.2 * max(1, num.shape[1])), sharex=True)
    plt.show()

In [ ]:
# Confusion matrix (key name may vary with the training script version)
cm_key = next((k for k in m if "confusion" in k), None)
if cm_key:
    cm = np.array(m[cm_key])
    names = labels["names"]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(names)), names, rotation=60, ha="right")
    ax.set_yticks(range(len(names)), names)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, v, ha="center", va="center", fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout(); plt.show()